In [ ]:
import os 
import getpass
import pydantic
from dotenv import load_dotenv
from crewai import LLM, Agent, Task, Crew
from chromadb.config import Settings
from crewai.rag.chromadb.config import ChromaDBConfig
from chromadb.config import Settings
from crewai.tools import tool

llm = LLM(model="ollama/llama3.2:1b-instruct-q8_0", base_url="http://localhost:11434")

In [ ]:
from crewai.knowledge.source.text_file_knowledge_source import TextFileKnowledgeSource

text_kw_source = TextFileKnowledgeSource(
    file_paths=[
        "designs.txt", 
        "machine_learning.txt",
        "python_intro.txt"
    ]
)

In [ ]:
from crewai import Agent, Crew, Process, Task
from crewai.project import CrewBase, agent, crew, task
from crewai_tools import SerperDevTool
from pydantic import BaseModel, Field
from typing import List, Dict


class ScanPoint(BaseModel):
    topic: str = Field(description="The main topic or area being discussed")
    summary: str = Field(description="The key findings or insights about this topic")
    benefits: List[Dict[str, str]] = Field(
        description="The key benifit and details for topic",
        default_factory=list
    )
    references: List[str] = Field(
        description="the key reference design diagram url for discussed topic",
        default_factory=list
    )

In [ ]:

# Agent city expert
scan_doc_agent = Agent(
    role="As a software architect with expertise in {topic}",
    goal="Design robust, scalable system architectures that balance performance, maintainability, and cost-effectiveness",
    backstory="""
    With 20+ years of experience building {topic} systems at scale, you've developed a pragmatic approach 
    to software architecture. You've seen both successful and failed systems and have learned valuable lessons from each. 
    You balance theoretical best practices with practical constraints and always consider the maintenance and operational 
    aspects of your designs.
    """,
    tools=[],
    verbose=True,
    knowledge_sources=[text_kw_source],
    embedder={
        "provider": "openai",
        "config": {
            "model": "nomic-embed-text",
            "api_key":"",
            "platform_url":"http://localhost:11434/v1"
        },
    },
    llm=llm,
    allow_delegation=False,
)


# Define Tasks
scan_doc_task = Task(
    description='Extract key Benefits for {topic} and provide a summarized report.',
    expected_output='Output should include topic benifit, summary. output should include references as list',
    agent=scan_doc_agent,
    output_json=ScanPoint,
    output_file="agentic-ai/crewai/desings/doc-summary.json"
)


# Review the context you got and expands each topic into full section for a report about {topic}
# Make sure you find top 10 interesting and relevant design url about {topic} and return list of url
# Define Tasks
format_doc_task = Task(
    description='Extract key Benefits for {topic} and provide a summarized report.',
    expected_output='Output should include topic benifit, summary. output should include references as list',
    agent=scan_doc_agent,
    output_json=ScanPoint,
    output_file="agentic-ai/crewai/desings/doc-summary.json"
)

In [ ]:

# Assemble the Crew
from crewai import Crew, Process
from datetime import datetime

crew = Crew(
    agents=[scan_doc_agent],
    tasks=[scan_doc_task],
    process=Process.sequential,
    full_output=True,
    share_crew=False,
    verbose=True
)

inputs = {
    'topic': 'Microservice Design',
    'date': datetime.now().strftime('%Y-%m-%d')
}

# Execute the Tasks
result = crew.kickoff(inputs=inputs)
print(result)

In [ ]:
from crewai_tools import RagTool
from crewai_tools.tools.rag import RagToolConfig, VectorDbConfig, ProviderSpec
from crewai.rag.embeddings.providers.openai.types import OpenAIProviderSpec
# Create a RAG tool with custom configuration
import chromadb
from chromadb.config import Settings

client = chromadb.PersistentClient(path="/home/brijeshdhaker/IdeaProjects/crewai_design_document/vactor_store/chroma")
vectordb: VectorDbConfig = {
    "provider": "chromadb",
    "config": {
        "collection_name": "sandbox_documents",
        "persist_directory":"/home/brijeshdhaker/IdeaProjects/crewai_design_document/vactor_store/chroma",
        "allow_reset":True, 
        "is_persistent":True
    }
}

embedding_model: OpenAIProviderSpec = {
    "provider": "openai",
    "config":{
        "model": "nomic-embed-text",
        "api_key":"",
        "platform_url":"http://localhost:11434/v1"
    }
}

config: RagToolConfig = {
    "vectordb": vectordb,
    "embedding_model": embedding_model,
}

rag_tool = RagTool(config=config, summarize=True)

r = rag_tool.run("JPMORGAN")
r

In [18]:
from crewai import Agent, Task, Crew, Process, LLM
from crewai.knowledge.storage.knowledge_storage import KnowledgeStorage
from crewai.knowledge.source.base_knowledge_source import BaseKnowledgeSource
import requests
from datetime import datetime
from typing import Dict, Any
from pydantic import BaseModel, Field
from com.example.agentic.embedding.EmbeddingManager import EmbeddingManager
import json
import chromadb
from chromadb.config import Settings
from crewai.utilities.paths import db_storage_path
import os

# Create custom storage with specific embedder
custom_storage = KnowledgeStorage(
    embedder={
        "provider":"ollama",
        "config": {
            "model": 
            "nomic-embed-text"
        }
    },
    collection_name="designs"
)


class DesignKnowledgeSource(BaseKnowledgeSource):
    """Knowledge source that fetches data from Space News API."""
    limit: int = Field(default=10, description="Number of document to fetch")

    def load_content(self) -> Dict[Any, str]:
        """Fetch and format space news articles."""
        try:
            # Load documents from external source
            json_articles = self.load_json()
            croma_articles = self.load_from_vactor_store("JPMORGAN")

            formatted_document = self.validate_content(json_articles)
            return {self.collection_name: formatted_document}
        except Exception as e:
            raise ValueError(f"Failed to fetch space news: {str(e)}")

    #
    def load_json(self) :
        articles = []
        try:
            work_dir = os.getenv("WORK_DIR")
            with open(f"{work_dir}/docs/json/articals.json", 'r') as file:
                data = json.load(file)
                articles = data.get('results', [])
        except FileNotFoundError:
            print("Error: The file was not found.")
        except json.JSONDecodeError:
            print("Error: Failed to decode JSON (invalid syntax).")
        
        return articles    

        #
    def load_from_vactor_store(self, query: str, top_k: int = 5, score_threshold: float = 0.0) :
        articles = []
        try:

            # Connect to CrewAI's knowledge ChromaDB
            #knowledge_path = os.path.join("/home/brijeshdhaker/IdeaProjects/crewai_design_document/", "knowledge")
            knowledge_path = "/home/brijeshdhaker/IdeaProjects/crewai_design_document/vactor_store/chroma"
            work_dir = os.getenv("WORK_DIR")
            
            if os.path.exists(knowledge_path):
                vactorstore = chromadb.PersistentClient(path=knowledge_path)
                collections = vactorstore.list_collections()
                
                print("Knowledge Collections:")
                for collection in collections:
                    print(f"  - {collection.name}: {collection.count()} documents")
                    
                    # Sample a few documents to verify content
                    if collection.count() > 0:
                        sample = collection.peek(limit=2)
                        print(f"Sample content: {sample['documents'][0]}...")
                    else:
                        print("No knowledge storage found")

            

            print(f"Retrieving documents for query: '{query}'")
            print(f"Top K: {top_k}, Score threshold: {score_threshold}")
            # Generate query embedding
            embedding_manager=EmbeddingManager()
            query_embedding = embedding_manager.embed_query([query])
            
            # Search in vector store
            try:
                collection = vactorstore.get_or_create_collection(
                    name="sandbox_documents",
                    metadata={"description": "Document Embeddings for RAG"}
                )
                print(f"Vector store initialized. Collection: {collection}")
                print(f"Existing documents in collection: {collection.count()}")

                results = collection.query(
                    query_embeddings=query_embedding,
                    n_results=top_k
                )
                
                # Process results
                retrieved_docs = []
                
                if results['documents'] and results['documents'][0]:
                    documents = results['documents'][0]
                    metadatas = results['metadatas'][0]
                    distances = results['distances'][0]
                    ids = results['ids'][0]
                    
                    for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                        # Convert distance to similarity score (ChromaDB uses cosine distance)
                        similarity_score = 1 - distance
                        
                        if similarity_score >= score_threshold:
                            retrieved_docs.append({
                                'id': doc_id,
                                'content': document,
                                'metadata': metadata,
                                'similarity_score': similarity_score,
                                'distance': distance,
                                'rank': i + 1
                            })
                    
                    print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
                else:
                    print("No documents found")
                
                return retrieved_docs
            
            except Exception as e:
                print(f"Error during retrieval: {e}")
                return []
        
        #
        except FileNotFoundError:
            print("Error: The file was not found.")
        except json.JSONDecodeError:
            print("Error: Failed to decode JSON (invalid syntax).")
        
        return articles  
    
    #    
    def validate_content(self, articles: list) -> str:
        """Format articles into readable text."""
        formatted = "Space News Articles:\n\n"
        for article in articles:
            formatted += f"""
                Title: {article['title']}
                Published: {article['published_at']}
                Summary: {article['summary']}
                News Site: {article['news_site']}
                URL: {article['url']}
                -------------------"""
        return formatted
    
    #
    def add(self) -> None:
        """Process and store the articles."""
        # if len(documents) == 0:
        #     raise ValueError("Number of documents must be greater then 0")
        #     print(f"Adding {len(self.documents)} documents to knowledge store...")
        # self.documents = documents
        content = self.load_content()
        for _, text in content.items():
            chunks = self._chunk_text(text)
            #print(self.get_embeddings)
            self.chunks.extend(chunks)

    #
    def aadd(self) -> None:
        pass

# Create knowledge source
recent_news = DesignKnowledgeSource(
    storage=custom_storage,
    limit=10,
)

recent_news.load_content()



Knowledge Collections:
  - sandbox_documents: 6 documents
Sample content: Skip to main content

CrewAI home page

light logo

dark logo

HomeDocumentationAMPAPI ReferenceExamplesChangelog

Website

Forum

Blog

CrewGPT

Welcome

CrewAI Documentation

Welcome

CrewAI Documentation

Build collaborative AI agents, crews, and flows — production ready from day one.

Ship multi‑agent systems with confidence

Design agents, orchestrate crews, and automate flows with guardrails, memory, knowledge, and observability baked in.

Get startedView changelogAPI Reference

Get started

Introduction

Overview of CrewAI concepts, architecture, and what you can build with agents, crews, and flows.

Installation

Install via uv, configure API keys, and set up the CLI for local development.

Quickstart

Spin up your first crew in minutes. Learn the core runtime, project layout, and dev loop.

Build the basics

Agents

Compose agents with tools, memory, knowledge, and structured outputs using Pydantic. Incl

/home/brijeshdhaker/IdeaProjects/crewai_design_document/src/main/py/com/example/agentic/embedding/EmbeddingManager.py:37: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Embedding dimension: {self.model.get_sentence_embedding_dimension()}")


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Generated embeddings with shape: (1, 384)
Vector store initialized. Collection: Collection(name=sandbox_documents)
Existing documents in collection: 6
Retrieved 0 documents (after filtering)


{None: 'Space News Articles:\n\n\n                Title: CSDA Quality Assessment Report Evaluates Satellogic NewSat Data\n                Published: 2026-04-17T21:21:02Z\n                Summary: The report adds to the growing documentation on commercial data’s contributions to Earth science research and applications.\n                News Site: NASA\n                URL: https://science.nasa.gov/uncategorized/csda-quality-assessment-report-evaluates-satellogic-newsat-data/\n                -------------------\n                Title: Webinar 4/29: NASA CSDA Program Vendor Focus- MDA Space\n                Published: 2026-04-17T20:33:51Z\n                Summary: Join us April 29 at 2:00 p.m. EDT to learn more about CSDA program vendor MDA Space.\n                News Site: NASA\n                URL: https://science.nasa.gov/science-research/earth-science/webinar-4-29-nasa-csda-program-vendor-focus-mda-space/\n                -------------------\n                Title: NASA Artemis II H

In [ ]:
# Create specialized agent
space_analyst = Agent(
    role="Space News Analyst",
    goal="Answer questions about space news accurately and comprehensively",
    backstory="""You are a space industry analyst with expertise in space exploration,
    satellite technology, and space industry trends. You excel at answering questions
    about space news and providing detailed, accurate information.""",
    knowledge_sources=[recent_news],
    llm= LLM(model="ollama/llama3.2:1b-instruct-q8_0", base_url="http://localhost:11434")
)

# Create task that handles user questions
analysis_task = Task(
    description="Answer this question about space news: {user_question}",
    expected_output="A detailed answer based on the recent space news articles",
    agent=space_analyst
)

# Create and run the crew
crew = Crew(
    agents=[space_analyst],
    tasks=[analysis_task],
    verbose=True,
    process=Process.sequential,
    embedder={
        "provider": "ollama",
        "config": {
            "model":"nomic-embed-text",
            "url": "http://localhost:11434/api/embeddings"
        }
    }
)

# Example usage
result = crew.kickoff(
    inputs={"user_question": "What are the latest developments in space exploration?"}
)

╭──────────────────────────────────────────── ✨ Update Available ✨ ─────────────────────────────────────────────╮
│                                                                                                                 │
│  A new version of CrewAI is available!                                                                          │
│                                                                                                                 │
│  Current version: 1.14.1                                                                                        │
│  Latest version:  1.14.2                                                                                        │
│                                                                                                                 │
│  To update, run: uv sync --upgrade-package crewai                                                               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 070c7ad3-2973-4008-9cea-2f559b97d5e0                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Retrieving documents for query: 'JPMORGAN'
Top K: 5, Score threshold: 0.0
Vector store initialized. Collection: Collection(name=sandbox_documents)
Existing documents in collection: 6
Error during retrieval: Expected Embeddings to be non-empty list or numpy array, got [] in query.


╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Answer this question about space news: What are the latest developments in space exploration?            │
│  ID: 36348c3b-3231-4815-9998-f234473ebf89                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────── 🔍 Knowledge Retrieval ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Knowledge Retrieval Started                                                                                    │
│  Status: Retrieving...                                                                                          │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Space News Analyst                                                                                      │
│                                                                                                                 │
│  Task: Answer this question about space news: What are the latest developments in space exploration?            │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Space News Analyst                                                                                      │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  The latest developments in space exploration involve various projects and initiatives that aim to push the     │
│  boundaries of what is possible in our solar system. Here are some of the key developments:                     │
│                                                                                                                 │
│  1. NASA's Heliophysics Audified: Resonances in Plasmas (HARP) citizen science project - The team behind        │
│  NASA's HARP compared the Earth's magnetic field to a giant harp in space, providing a unique approach to       │
│  understanding solar and cosmic waves.                                                                          │
│                                                                                                                 │
│  2. NASA Artemis II Human Research Data Methodology Challenge - NASA's Human Research Program (HRP) is using    │
│  various facilities, including the International Space Station and analog environments, to monitor human        │
│  health in deep space for future human exploration of Mars.                                                     │
│                                                                                                                 │
│  3. The new EU Space Act draft was seen as a step backward - A revised draft of proposed provisions in          │
│  European Union law governing space development is being debated, raising concerns about how the law would be   │
│  applied outside the EU.                                                                                        │
│                                                                                                                 │
│  4. CSDA Quality Assessment Report Evaluates Satellogic NewSat Data - The Climate Systems Data Analysis (CSDA)  │
│  program used data from commercial providers like Satellogic to assess its quality and accuracy in support of   │
│  Earth science research and applications.                                                                       │
│                                                                                                                 │
│  5. Webinar 4/29: NASA CSDA Program Vendor Focus- MDA Space - The webinar provided an opportunity for           │
│  attendees to learn more about the CSDA program vendor MDA Space, focusing on their services as part of NASA's  │
│  Artemis II mission.                                                                                            │
│                                                                                                                 │
│  6. NASA Artemis II pilot describes landing in Orion: "From intense to pure elation" - An astronaut described   │
│  his experience during the Orion spacecraft's landing on the Moon, from intense sensations of weightlessness    │
│  to feelings of pure elation after completing a successful lunar landing.                                       │
│                                                                                                                 │
│  7. Quality Assessment Report Evaluates Tomorrow.io Precipitation Radar Data - The report analyzed commercial   │
│  data from commercial providers like Tomorrow.io's precipitation radar and evaluated its contribution to Earth  │
│  science research and applications.                    

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Answer this question about space news: What are the latest developments in space exploration?            │
│  Agent: Space News Analyst                                                                                      │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── Tracing Preference Saved ────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing has been disabled.                                                                               │
│                                                                                                                 │
│  Your preference has been saved. Future Crew/Flow executions will not collect traces.                           │
│                                                                                                                 │
│  To enable tracing later, do any one of these:                                                                  │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 070c7ad3-2973-4008-9cea-2f559b97d5e0                                                                       │
│  Final Output: The latest developments in space exploration involve various projects and initiatives that aim   │
│  to push the boundaries of what is possible in our solar system. Here are some of the key developments:         │
│                                                                                                                 │
│  1. NASA's Heliophysics Audified: Resonances in Plasmas (HARP) citizen science project - The team behind        │
│  NASA's HARP compared the Earth's magnetic field to a giant harp in space, providing a unique approach to       │
│  understanding solar and cosmic waves.                                                                          │
│                                                                                                                 │
│  2. NASA Artemis II Human Research Data Methodology Challenge - NASA's Human Research Program (HRP) is using    │
│  various facilities, including the International Space Station and analog environments, to monitor human        │
│  health in deep space for future human exploration of Mars.                                                     │
│                                                                                                                 │
│  3. The new EU Space Act draft was seen as a step backward - A revised draft of proposed provisions in          │
│  European Union law governing space development is being debated, raising concerns about how the law would be   │
│  applied outside the EU.                                                                                        │
│                                                                                                                 │
│  4. CSDA Quality Assessment Report Evaluates Satellogic NewSat Data - The Climate Systems Data Analysis (CSDA)  │
│  program used data from commercial providers like Satellogic to assess its quality and accuracy in support of   │
│  Earth science research and applications.                                                                       │
│                                                                                                                 │
│  5. Webinar 4/29: NASA CSDA Program Vendor Focus- MDA Space - The webinar provided an opportunity for           │
│  attendees to learn more about the CSDA program vendor MDA Space, focusing on their services as part of NASA's  │
│  Artemis II mission.                                                                                            │
│                                                                                                                 │
│  6. NASA Artemis II pilot describes landing in Orion: "From intense to pure elation" - An astronaut described   │
│  his experience during the Orion spacecraft's landing on the Moon, from intense sensations of weightlessness    │
│  to feelings of pure elation after completing a successful lunar landing.                                       │
│                                                                                                                 │
│  7. Quality Assessment Report Evaluates Tomorrow.io Precipitation Radar Data - The report analyzed commercial   │
│  data from commercial providers like Tomorrow.io's precipitation radar and evaluated its contribution to Earth  │
│  science research and applications.                   